# Example 15.5 / 1: Debug an Error with the Server

Companion notebook for **15.5 AMPL MCP Server** in the AMPL lab Documentation, section "Debug an Error with the Server". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Debug an Error with the Server

A debug-mode search starts from the exact message. In this failing input, the model fixes
`avail` with `=`, and the script then tries to assign it:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;               # steel products
param rate {PROD} > 0;  # tons per hour
param avail = 40;       # fixed by = in the model; data cannot replace it
var Make {PROD} >= 0;   # tons produced

maximize Total_Tons: sum {p in PROD} Make[p];  # total output, tons

# rolling hours used fit within avail
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # data.py
    ampl.set["PROD"] = ["bands", "coils"]
    ampl.param["rate"] = {"bands": 200, "coils": 140}
    ampl.param["avail"] = 35
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "let" command:

	avail has an = assignment in the model.
